# Movie Recommender Analysis: Normalizing and Standardizing Ratings

The objective of this notebook is to process, normalize, and standardize movie ratings collected from a cohort of reviewers across a selection of recent popular and classic movies. When analyzing raw ratings, individual reviewer biases (e.g., some people always rate high, others always rate low) can distort overall statistics. Normalization and standardization help balance these baselines for fairer comparisons.

### Movie Selection & Survey Baseline
I chose the following eight movies for this analysis:
1. Chainsaw Man – The Movie: Reze Arc
2. Akira
3. Spirited Away
4. Mickey Blue Eyes
5. The Sixth Sense
6. The Equalizer (2014)
7. Hellboy
8. Hellboy II: The Golden Army

To capture realistic missing data (as not everyone has seen every movie), several reviews contain null/NaN values. Five imaginary reviewers (Alex, Bob, Charlie, Dana, Eli) provided their ratings on a scale of 1 to 5.

In [5]:
import pandas as pd
import numpy as np

# Create raw dataset
# Movie names are columns at the top, imaginary names on the left (index)
raw_data = {
    'Reviewer': ['Alex', 'Bob', 'Charlie', 'Dana', 'Eli'],
    'Chainsaw Man – The Movie: Reze Arc': [5.0, np.nan, 4.0, 3.0, np.nan],
    'Akira': [4.0, 5.0, np.nan, 3.0, 4.0],
    'Spirited Away': [5.0, 4.0, 5.0, 4.0, 5.0],
    'Mickey Blue Eyes': [2.0, 3.0, 3.0, np.nan, 2.0],
    'The Sixth Sense': [4.0, 4.0, np.nan, 5.0, 4.0],
    'The Equalizer (2014)': [3.0, np.nan, 4.0, 3.0, 3.0],
    'Hellboy': [3.0, 3.0, 2.0, 4.0, np.nan],
    'Hellboy II: The Golden Army': [np.nan, 3.0, 3.0, 4.0, 3.0]
}

df_raw = pd.DataFrame(raw_data).set_index('Reviewer')

# Save to CSV and reload to demonstrate load workflow
df_raw.to_csv('movie_ratings.csv')
df_ratings = pd.read_csv('movie_ratings.csv', index_col='Reviewer')

print("=== Raw Movie Ratings Table ===")
display(df_ratings.fillna(''))

=== Raw Movie Ratings Table ===


,Chainsaw Man – The Movie: Reze Arc,Akira,Spirited Away,Mickey Blue Eyes,The Sixth Sense,The Equalizer (2014),Hellboy,Hellboy II: The Golden Army
Reviewer,,,,,,,,
Alex,5.0,4.0,5.0,2.0,4.0,3.0,3.0,
Bob,,5.0,4.0,3.0,4.0,,3.0,3.0
Charlie,4.0,,5.0,3.0,,4.0,2.0,3.0
Dana,3.0,3.0,4.0,,5.0,3.0,4.0,4.0
Eli,,4.0,5.0,2.0,4.0,3.0,,3.0


### Part 1 & 2: Average Ratings per User and Movie

We calculate the average rating given by each reviewer (user) across the movies they have watched, and the average rating received by each movie from all who watched it.

In [2]:
# Calculate averages
user_means = df_ratings.mean(axis=1).round(2)
movie_means = df_ratings.mean(axis=0).round(2)

print("=== Average Ratings Given by Each User ===")
display(user_means.to_frame(name='Average Rating Given'))

print("\n=== Average Ratings Received by Each Movie ===")
display(movie_means.to_frame(name='Average Rating Received'))

=== Average Ratings Given by Each User ===


,Average Rating Given
Reviewer,
Alex,3.71
Bob,3.67
Charlie,3.50
Dana,3.71
Eli,3.50



=== Average Ratings Received by Each Movie ===


,Average Rating Received
Chainsaw Man – The Movie: Reze Arc,4.00
Akira,4.00
Spirited Away,4.60
Mickey Blue Eyes,2.50
The Sixth Sense,4.25
The Equalizer (2014),3.25
Hellboy,3.00
Hellboy II: The Golden Army,3.25


### Part 3: Normalized Ratings

Normalization scales values into a relative $[0, 1]$ range based on the minimum and maximum scores given by each specific user.

$$\text{Normalized Rating} = \frac{R - R_{\text{min}}}{R_{\text{max}} - R_{\text{min}}}$$

If a user rates all movies the same, the range is zero; we will handle this by returning 0.5 (neutral) or leaving it unchanged to avoid division by zero.

In [6]:
# Normalize ratings for each user
user_min = df_ratings.min(axis=1)
user_max = df_ratings.max(axis=1)

# Prevent division by zero if min equals max
user_range = user_max - user_min
user_range = user_range.replace(0, 1)

df_normalized = df_ratings.sub(user_min, axis=0).div(user_range, axis=0)

print("=== Normalized Ratings Table ===")
display(df_normalized.fillna(''))

# Calculate and display averages for normalized data
norm_user_means = df_normalized.mean(axis=1).round(2)
norm_movie_means = df_normalized.mean(axis=0).round(2)

print("\n=== Average Normalized Rating Given by Each User ===")
display(norm_user_means.to_frame(name='Average Normalized Rating Given'))

print("\n=== Average Normalized Rating Received by Each Movie ===")
display(norm_movie_means.to_frame(name='Average Normalized Rating Received'))

=== Normalized Ratings Table ===


,Chainsaw Man – The Movie: Reze Arc,Akira,Spirited Away,Mickey Blue Eyes,The Sixth Sense,The Equalizer (2014),Hellboy,Hellboy II: The Golden Army
Reviewer,,,,,,,,
Alex,1.0,0.666667,1.0,0.0,0.666667,0.333333,0.333333,
Bob,,1.0,0.5,0.0,0.5,,0.0,0.0
Charlie,0.666667,,1.0,0.333333,,0.666667,0.0,0.333333
Dana,0.0,0.0,0.5,,1.0,0.0,0.5,0.5
Eli,,0.666667,1.0,0.0,0.666667,0.333333,,0.333333



=== Average Normalized Rating Given by Each User ===


,Average Normalized Rating Given
Reviewer,
Alex,0.57
Bob,0.33
Charlie,0.50
Dana,0.36
Eli,0.50



=== Average Normalized Rating Received by Each Movie ===


,Average Normalized Rating Received
Chainsaw Man – The Movie: Reze Arc,0.56
Akira,0.58
Spirited Away,0.80
Mickey Blue Eyes,0.08
The Sixth Sense,0.71
The Equalizer (2014),0.33
Hellboy,0.21
Hellboy II: The Golden Army,0.29


### Part 4: Analytical Discussion and Conclusions

#### Advantages of Normalized Ratings
- **Corrects for User Bias (Severity/Leniency):** Optimistic users who rate everything between 4 and 5, and pessimistic users who rate everything between 1 and 2, are put on the same scale. A 5 from an optimist and a 2 from a pessimist both represent their peak preference and will both scale to 1.0.
- **Easier Comparisons:** Brings all ratings into a bounded, easily understandable interval $[0, 1]$, which is also especially useful for recommendation algorithms.

#### Disadvantages of Normalized Ratings
- **Loss of Absolute Scale:** An absolute rating of 5 represents stellar quality, while 3 represents mediocre quality. Normalization removes this context; if a user's ratings only range from 2 to 3, then a 3 becomes a 1.0 (perfect score) while a 2 becomes 0.0 (worst score).
- **Sensitivity to Outliers:** A single extreme rating can heavily compress or distort the rest of the normalized values for that user.

### Part 5: Standardized Ratings

Standardization (Z-score normalization) scales the ratings by subtracting the user's mean and dividing by their standard deviation. This shifts the distribution to have a mean of 0 and a standard deviation of 1.

$$\text{Standardized Rating} = \frac{R - \mu}{\sigma}$$

Where $\mu$ is the user's mean rating and $\sigma$ is the user's rating standard deviation.

In [7]:
# Standardize ratings for each user
user_mean_val = df_ratings.mean(axis=1)
user_std_val = df_ratings.std(axis=1)

# Prevent division by zero if std is 0 or NaN
user_std_val = user_std_val.replace(0, 1).fillna(1)

df_standardized = df_ratings.sub(user_mean_val, axis=0).div(user_std_val, axis=0)

print("=== Standardized Ratings Table ===")
display(df_standardized.fillna(''))

# Calculate and display averages for standardized data
std_user_means = df_standardized.mean(axis=1).round(2)
std_movie_means = df_standardized.mean(axis=0).round(2)

print("\n=== Average Standardized Rating Given by Each User ===")
display(std_user_means.to_frame(name='Average Standardized Rating Given'))

print("\n=== Average Standardized Rating Received by Each Movie ===")
display(std_movie_means.to_frame(name='Average Standardized Rating Received'))

=== Standardized Ratings Table ===


,Chainsaw Man – The Movie: Reze Arc,Akira,Spirited Away,Mickey Blue Eyes,The Sixth Sense,The Equalizer (2014),Hellboy,Hellboy II: The Golden Army
Reviewer,,,,,,,,
Alex,1.155493,0.256776,1.155493,-1.540658,0.256776,-0.641941,-0.641941,
Bob,,1.632993,0.408248,-0.816497,0.408248,,-0.816497,-0.816497
Charlie,0.476731,,1.430194,-0.476731,,0.476731,-1.430194,-0.476731
Dana,-0.944911,-0.944911,0.377964,,1.70084,-0.944911,0.377964,0.377964
Eli,,0.476731,1.430194,-1.430194,0.476731,-0.476731,,-0.476731



=== Average Standardized Rating Given by Each User ===


,Average Standardized Rating Given
Reviewer,
Alex,-0.0
Bob,0.0
Charlie,0.0
Dana,-0.0
Eli,0.0



=== Average Standardized Rating Received by Each Movie ===


,Average Standardized Rating Received
Chainsaw Man – The Movie: Reze Arc,0.23
Akira,0.36
Spirited Away,0.96
Mickey Blue Eyes,-1.07
The Sixth Sense,0.71
The Equalizer (2014),-0.40
Hellboy,-0.63
Hellboy II: The Golden Army,-0.35
